# Cell 1: Setup — clone repo + add source dirs to path
import sys, os, time, json
from collections import Counter, defaultdict

REPO_URL = "https://github.com/adriel007/ternary-boost.git"
ROOT = "ternary-boost"

if not os.path.exists(ROOT):
    !git clone {REPO_URL}
%cd {ROOT}
!pip install -q transformers safetensors datasets pandas

for m in ["shared", "pt_bitnet", "paretoq", "tequila", "eval", "chat"]:
    sys.path.insert(0, os.path.join(m, "src"))

import torch, torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
print(f"Device: {device} | Dtype: {dtype}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    mem_gb = getattr(gpu, 'total_memory', getattr(gpu, 'total_mem', 0)) / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {mem_gb:.1f} GB")

In [ ]:
# Cell 1: Environment setup
import sys, os, time, json
from collections import Counter, defaultdict

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    !pip install -q torch transformers safetensors datasets
    # Clone repo if not present
    if not os.path.exists('ternary-boost'):
        !git clone https://github.com/user/ternary-boost.git
    %cd ternary-boost
    for m in ["shared", "pt_bitnet", "paretoq", "tequila", "eval", "chat"]:
        sys.path.insert(0, f"{m}/src")
else:
    # Local: add src dirs
    for m in ["shared", "pt_bitnet", "paretoq", "tequila", "eval", "chat"]:
        sys.path.insert(0, f"{m}/src")

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
print(f"Device: {device} | Dtype: {dtype}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

In [ ]:
# Cell 2: Load base model + calibration data
MODEL_ID = "microsoft/phi-2"

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def load_fresh_model():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True,
        device_map=device, trust_remote_code=True)

# Calibration texts
try:
    from datasets import load_dataset
    wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    calib_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:64]
except:
    calib_texts = ["The capital of France is Paris. " * 5] * 32

print(f"Calibration texts: {len(calib_texts)}")

# Eval texts for perplexity
eval_texts = calib_texts[48:64]  # Use separate split
print(f"Eval texts: {len(eval_texts)}")

In [ ]:
# Cell 3: Evaluation utilities
def compute_perplexity(model, tokenizer, texts, max_length=256):
    """Compute perplexity on a set of texts."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
            input_ids = enc["input_ids"].to(device)
            if input_ids.shape[1] < 2:
                continue
            outputs = model(input_ids, labels=input_ids)
            loss = outputs.loss
            if loss is not None:
                total_loss += loss.item() * input_ids.shape[1]
                total_tokens += input_ids.shape[1]
    return torch.exp(torch.tensor(total_loss / max(total_tokens, 1))).item()

def measure_speed(model, tokenizer, prompt, max_tokens=30, runs=3):
    """Measure generation speed."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)
    times = []
    tok_counts = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.time()
            out = model.generate(input_ids, max_new_tokens=max_tokens,
                                do_sample=True, temperature=0.7,
                                pad_token_id=tokenizer.eos_token_id)
            elapsed = time.time() - t0
            times.append(elapsed)
            tok_counts.append(out.shape[1] - input_ids.shape[1])
    avg_tokens = sum(tok_counts) / len(tok_counts)
    avg_time = sum(times) / len(times)
    return avg_tokens / avg_time if avg_time > 0 else 0

def weight_sparsity(model):
    """Measure weight sparsity (fraction of near-zero weights)."""
    total = 0
    zeros = 0
    for name, param in model.named_parameters():
        if "weight" in name and "embed" not in name and "lm_head" not in name:
            w = param.data
            total += w.numel()
            zeros += (w.abs() < 1e-6).sum().item()
    return zeros / max(total, 1)

def unique_per_row(model):
    """Average unique values per row (should be ~3 for ternary)."""
    counts = []
    for name, param in model.named_parameters():
        if "weight" in name and "proj" in name:
            w = param.data
            for i in range(min(w.shape[0], 20)):
                counts.append(w[i].unique().numel())
    return sum(counts) / len(counts) if counts else 0

def generate_sample(model, tokenizer, prompt):
    """Generate a sample for quality inspection."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=30, temperature=0.7,
                            do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

print("Evaluation utilities ready.")

In [ ]:
# Cell 4: BASELINE — FP16 original model
print("=" * 60)
print("BASELINE: FP16 Original Model")
print("=" * 60)

model_base = load_fresh_model()
results = {}

results["base"] = {
    "perplexity": compute_perplexity(model_base, tokenizer, eval_texts),
    "speed_tok_s": measure_speed(model_base, tokenizer, "The capital of France is"),
    "sparsity": weight_sparsity(model_base),
    "unique_per_row": unique_per_row(model_base),
    "params": sum(p.numel() for p in model_base.parameters()),
    "sample": generate_sample(model_base, tokenizer, "What is the capital of France?"),
}

print(f"Perplexity:     {results['base']['perplexity']:.2f}")
print(f"Speed:          {results['base']['speed_tok_s']:.2f} tok/s")
print(f"Sparsity:       {results['base']['sparsity']*100:.1f}%")
print(f"Unique/row:     {results['base']['unique_per_row']:.1f}")
print(f"Sample:         '{results['base']['sample'].strip()[:100]}'")
del model_base

In [ ]:
# Cell 5: VARIANT A — PT-BitNet (pure ternary, no outliers)
from pt_bitnet import apply_pt_bitnet, PTBitNetConfig

print("=" * 60)
print("VARIANT A: PT-BitNet (pure ternary)")
print("=" * 60)

model_a = load_fresh_model()
cfg_a = PTBitNetConfig(outlier_fraction=0.0, compensation_steps=0, show_progress=True)
model_a = apply_pt_bitnet(model_a, cfg_a)

results["pt_bitnet"] = {
    "perplexity": compute_perplexity(model_a, tokenizer, eval_texts),
    "speed_tok_s": measure_speed(model_a, tokenizer, "The capital of France is"),
    "sparsity": weight_sparsity(model_a),
    "unique_per_row": unique_per_row(model_a),
    "params": sum(p.numel() for p in model_a.parameters()),
    "sample": generate_sample(model_a, tokenizer, "What is the capital of France?"),
}

print(f"Perplexity:     {results['pt_bitnet']['perplexity']:.2f}")
print(f"Speed:          {results['pt_bitnet']['speed_tok_s']:.2f} tok/s")
print(f"Sparsity:       {results['pt_bitnet']['sparsity']*100:.1f}%")
print(f"Unique/row:     {results['pt_bitnet']['unique_per_row']:.1f}")
print(f"Sample:         '{results['pt_bitnet']['sample'].strip()[:100]}'")
del model_a

In [ ]:
# Cell 6: VARIANT B — PT-BitNet + Outlier Retention (SpQR-style)

print("=" * 60)
print("VARIANT B: PT-BitNet + 1% Outlier Retention")
print("=" * 60)

model_b = load_fresh_model()
cfg_b = PTBitNetConfig(outlier_fraction=0.01, compensation_steps=0, show_progress=True)
model_b = apply_pt_bitnet(model_b, cfg_b)

results["outliers"] = {
    "perplexity": compute_perplexity(model_b, tokenizer, eval_texts),
    "speed_tok_s": measure_speed(model_b, tokenizer, "The capital of France is"),
    "sparsity": weight_sparsity(model_b),
    "unique_per_row": unique_per_row(model_b),
    "params": sum(p.numel() for p in model_b.parameters()),
    "sample": generate_sample(model_b, tokenizer, "What is the capital of France?"),
}

print(f"Perplexity:     {results['outliers']['perplexity']:.2f}")
print(f"Speed:          {results['outliers']['speed_tok_s']:.2f} tok/s")
print(f"Sparsity:       {results['outliers']['sparsity']*100:.1f}%")
print(f"Unique/row:     {results['outliers']['unique_per_row']:.1f}")
print(f"Sample:         '{results['outliers']['sample'].strip()[:100]}'")
del model_b

In [ ]:
# Cell 7: VARIANT C — PT-BitNet + Outliers + Hessian Compensation

print("=" * 60)
print("VARIANT C: PT-BitNet + Outliers + Hessian Compensation")
print("=" * 60)

model_c = load_fresh_model()
cfg_c = PTBitNetConfig(outlier_fraction=0.01, compensation_steps=50, show_progress=True)
model_c = apply_pt_bitnet(model_c, cfg_c, tokenizer=tokenizer, calibration_texts=calib_texts[:32])

results["compensation"] = {
    "perplexity": compute_perplexity(model_c, tokenizer, eval_texts),
    "speed_tok_s": measure_speed(model_c, tokenizer, "The capital of France is"),
    "sparsity": weight_sparsity(model_c),
    "unique_per_row": unique_per_row(model_c),
    "params": sum(p.numel() for p in model_c.parameters()),
    "sample": generate_sample(model_c, tokenizer, "What is the capital of France?"),
}

print(f"Perplexity:     {results['compensation']['perplexity']:.2f}")
print(f"Speed:          {results['compensation']['speed_tok_s']:.2f} tok/s")
print(f"Sparsity:       {results['compensation']['sparsity']*100:.1f}%")
print(f"Unique/row:     {results['compensation']['unique_per_row']:.1f}")
print(f"Sample:         '{results['compensation']['sample'].strip()[:100]}'")
del model_c

In [ ]:
# Cell 8: RESULTS — Summary table
import pandas as pd

rows = []
for name, r in results.items():
    base_ppl = results["base"]["perplexity"]
    ppl_delta = r["perplexity"] - base_ppl
    rows.append({
        "Variant": name,
        "Perplexity": f"{r['perplexity']:.2f}",
        "Δ PPL": f"{ppl_delta:+.2f}",
        "Speed (tok/s)": f"{r['speed_tok_s']:.2f}",
        "Sparsity (%)": f"{r['sparsity']*100:.1f}",
        "Uniq/row": f"{r['unique_per_row']:.1f}",
        "Params (M)": f"{r['params']/1e6:.0f}",
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("ABLATION RESULTS")
print("=" * 80)
print(df.to_string(index=False))

print("\n" + "=" * 80)
print("GENERATION SAMPLES (prompt: 'What is the capital of France?')")
print("=" * 80)
for name, r in results.items():
    print(f"\n{name:20s}: {r['sample'].strip()[:120]}")

print("\n" + "=" * 80)
print("CONCLUSION")
print("=" * 80)
base = results['base']
best = min(results.items(), key=lambda x: x[1]['perplexity'])
print(f"Best Perplexity: {best[0]} ({best[1]['perplexity']:.2f}, Δ={best[1]['perplexity']-base['perplexity']:+.2f})")

if 'outliers' in results and 'pt_bitnet' in results:
    delta = results['outliers']['perplexity'] - results['pt_bitnet']['perplexity']
    print(f"Outlier retention gain: {delta:+.2f} PPL vs pure ternary")

if 'compensation' in results and 'outliers' in results:
    delta = results['compensation']['perplexity'] - results['outliers']['perplexity']
    print(f"Hessian compensation gain: {delta:+.2f} PPL vs outliers-only")

# Save results
with open('ablation_results.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'sample'} for k, v in results.items()}, f, indent=2)
print("\nResults saved to ablation_results.json")